# Day 37: Data Pipelines

**Today's question:** a messy data file goes in, a clean report comes out. How do we organise that program so
every part can be checked on its own, and a bug can be found in minutes instead of hours?

No new syntax today: only functions, files, dicts, exceptions and CSV/JSON, combined well.

In [ ]:
import csv
import json
from pathlib import Path

grades_path = Path("data/grades.csv")
out_dir = Path("output")

## 1. The messy file

Real data is never clean. 🤔 **Spot the problems** before we write any code.

In [ ]:
print(grades_path.read_text(encoding="utf-8"))

Problems we can **fix**: extra spaces and wrong capitals (`" asha khan"`, `Science`, `MATHS`).

Problems we **cannot fix**: a missing score, `absent` instead of a number, a score of `105`. We must not guess
those, so we **reject** the row and **say why**. Silently dropping rows hides problems.

## 2. The plan: a pipeline of four stages

```
grades.csv ─load─▶ raw rows ─clean─▶ clean rows ─analyse─▶ summary ─report─▶ text + JSON
```

| Stage | Function(s) | Takes | Gives back |
|-------|-------------|-------|------------|
| load | `load_rows(path)` | a file path | list of dicts (all text) |
| clean | `clean_row(row)`, `clean_rows(rows)` | raw rows | clean rows + list of problems |
| analyse | `analyse(clean)` | clean rows | a summary dict |
| report | `build_report(summary, problems)` | summary + problems | one string |

Rules: each stage **returns** its result (no printing inside), takes everything it needs as **parameters**, and
does **one** job. Only `main()` reads files, prints and saves.

## 3. Stage 1: load

In [ ]:
def load_rows(path):
    """Read a CSV file and return its rows as a list of dicts (all values are text)."""
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


rows = load_rows(grades_path)
print(len(rows), "rows")
print(rows[2])

Check each stage **as soon as it is written**: right number of rows? Does a row look as expected? Only then
move on.

## 4. Stage 2: clean

First, clean **one** row. Fixable problems get fixed; unfixable ones raise a `ValueError` that explains why.

In [ ]:
def is_number(text):
    """Return True if text can be converted to a float."""
    try:
        float(text)
        return True
    except ValueError:
        return False


def clean_row(row):
    """Return a tidy copy of one raw row, or raise ValueError saying what is wrong."""
    name = row["student"].strip().title()
    subject = row["subject"].strip().lower()
    score_text = row["score"].strip()
    if name == "":
        raise ValueError("missing student name")
    if score_text == "":
        raise ValueError("missing score")
    if not is_number(score_text):
        raise ValueError(f"score is not a number: '{score_text}'")
    score = float(score_text)
    if score < 0 or score > 100:
        raise ValueError(f"score out of range: {score_text}")
    return {"student": name, "subject": subject, "score": score}

We test it with **tiny hand-made rows**. No file needed: that is the benefit of a separate stage.

In [ ]:
print(clean_row({"student": " asha khan", "subject": "English ", "score": "91"}))

🤔 **Predict:** what happens with the score `"absent"`?

In [ ]:
clean_row({"student": "Omar Farouk", "subject": "maths", "score": "absent"})

Raising is `clean_row`'s job. *Deciding what to do* about a bad row belongs to the caller. Let's turn our checks
into `assert`s (Day 29) so we can rerun them after every change:

In [ ]:
def is_rejected(row):
    """Return True if clean_row refuses this row."""
    try:
        clean_row(row)
    except ValueError:
        return True
    return False


assert clean_row({"student": "mei chen ", "subject": "MATHS", "score": "88"}) == {
    "student": "Mei Chen", "subject": "maths", "score": 88.0}
assert is_rejected({"student": "Mei Chen", "subject": "maths", "score": "105"})
assert is_rejected({"student": "Bilal Ahmed", "subject": "science", "score": ""})
assert not is_rejected({"student": "Omar Farouk", "subject": "english", "score": "52.5"})
print("clean_row checks passed")

Now the whole list. `clean_rows` keeps the good rows and **collects** the problems with their line number in the file.

In [ ]:
def clean_rows(rows):
    """Split raw rows into (clean_rows, problems). Each problem names its line in the file."""
    clean = []
    problems = []
    for line_number, row in enumerate(rows, start=2):   # line 1 of the file is the header
        try:
            clean.append(clean_row(row))
        except ValueError as error:
            problems.append(f"line {line_number}: {error}")
    return clean, problems

🤔 **Predict:** of the 21 rows, how many survive?

In [ ]:
clean, problems = clean_rows(rows)
print(len(clean), "clean rows")
for problem in problems:
    print(problem)

## 5. Stage 3: analyse

Small helpers first, each easy to check.

In [ ]:
def letter_grade(average):
    """Turn an average score (0-100) into a letter grade."""
    if average >= 90:
        return "A"
    elif average >= 80:
        return "B"
    elif average >= 70:
        return "C"
    elif average >= 60:
        return "D"
    return "F"


def average(numbers):
    """Return the mean of a non-empty list of numbers."""
    return sum(numbers) / len(numbers)


def group_scores(clean, column):
    """Group scores by one column: {value_of_column: [score, score, ...]}."""
    groups = {}
    for row in clean:
        key = row[column]
        if key not in groups:
            groups[key] = []
        groups[key].append(row["score"])
    return groups


print(group_scores(clean, "subject")["maths"])

In [ ]:
def student_average(item):
    """Sort key for (name, info) pairs: the student's average."""
    return item[1]["average"]


def analyse(clean):
    """Return a summary dict: per-student results, subject averages, ranking, class average."""
    students = {}
    for name, scores in group_scores(clean, "student").items():
        student_avg = round(average(scores), 2)
        students[name] = {"average": student_avg, "grade": letter_grade(student_avg), "subjects": len(scores)}

    subjects = {}
    for subject, scores in group_scores(clean, "subject").items():
        subjects[subject] = round(average(scores), 2)

    ranked = sorted(students.items(), key=student_average, reverse=True)
    all_scores = [row["score"] for row in clean]
    return {
        "students": students,
        "subjects": subjects,
        "ranking": [name for name, info in ranked],
        "class_average": round(average(all_scores), 2),
    }

Test `analyse` on data so small you can do the maths **in your head** before trusting it with the real file.
🤔 **Predict** Ken's average, the maths average and the ranking.

In [ ]:
tiny = [
    {"student": "Ken Ito", "subject": "maths", "score": 80.0},
    {"student": "Ken Ito", "subject": "science", "score": 90.0},
    {"student": "Zara Malik", "subject": "maths", "score": 60.0},
]
tiny_summary = analyse(tiny)
print(tiny_summary)
assert tiny_summary["students"]["Ken Ito"]["average"] == 85.0
assert tiny_summary["subjects"]["maths"] == 70.0
assert tiny_summary["ranking"] == ["Ken Ito", "Zara Malik"]

In [ ]:
summary = analyse(clean)
print(summary["ranking"])
print(summary["subjects"])

## 6. Stage 4: report, then `main()` connects everything

`build_report` **returns** a string. Whoever calls it decides: print it, save it, or both.

In [ ]:
def build_report(summary, problems):
    """Return the whole report as one string. The caller decides whether to print or save it."""
    lines = ["GRADE REPORT", "=" * 38, "Rank  Student          Average  Grade"]
    for rank, name in enumerate(summary["ranking"], start=1):
        info = summary["students"][name]
        lines.append(f"{rank:<6}{name:<16}{info['average']:>8.2f}  {info['grade']}")
    lines.append("")
    lines.append("Subject averages")
    for subject, subject_avg in summary["subjects"].items():
        lines.append(f"  {subject:<10}{subject_avg:>7.2f}")
    lines.append(f"Class average: {summary['class_average']:.2f}")
    lines.append("")
    lines.append(f"Rows skipped ({len(problems)}):")
    for problem in problems:
        lines.append(f"  {problem}")
    return "\n".join(lines)


def save_text(text, path):
    """Save text to a file, creating its folder if needed."""
    path.parent.mkdir(exist_ok=True)
    path.write_text(text, encoding="utf-8")


def save_json(data, path):
    """Save data as pretty-printed JSON, creating the folder if needed."""
    path.parent.mkdir(exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

`main()` reads like the plan from section 2. If you can read `main()` aloud as a sentence, the structure is good.

In [ ]:
def main(data_path=grades_path):
    """Run the pipeline: load -> clean -> analyse -> report."""
    try:
        rows = load_rows(data_path)
    except FileNotFoundError:
        print(f"Cannot find {data_path}. Run this from the day-37 folder.")
        return
    clean, problems = clean_rows(rows)
    summary = analyse(clean)
    report = build_report(summary, problems)
    print(report)
    save_text(report, out_dir / "grade_report.txt")
    save_json(summary, out_dir / "grade_summary.json")


main()

🤔 **Predict:** what happens if the file is missing?

In [ ]:
main(Path("data/grades_2027.csv"))

## 7. Common mistakes (these cells fail **on purpose**)

**Mistake 1: a stage that prints instead of returning.** The next stage receives `None`.

In [ ]:
def average_printing(numbers):
    print(sum(numbers) / len(numbers))


result = average_printing([80, 90])
print(result + 5)

**Mistake 2: forgetting that a stage returns two values.** `clean_rows` returns a *tuple* `(clean, problems)`.

In [ ]:
summary = analyse(clean_rows(rows))

**Mistake 3 (no error message, which makes it worse!): a stage that changes its input.**

In [ ]:
def shout_names(rows):
    for row in rows:
        row["student"] = row["student"].upper()
    return rows


originals = [{"student": "Sara Lopez"}]
shouted = shout_names(originals)
print(originals)

The caller's list changed too (Day 07: two names, one list). That is why `clean_row` builds a **new** dict
instead of editing `row`. A stage should give back new data and leave its input alone.

## 8. From notebook to script

The same functions live in **`grade_pipeline.py`** in this folder, with `main()` called under
`if __name__ == "__main__":`, and **`test_grade_pipeline.py`** has one pytest test per stage. In the terminal,
from this folder:

```
uv run python grade_pipeline.py
uv run pytest -q
```

## Recap

1. A pipeline is a chain of stages: **load → clean → analyse → report**. Each is a small function with clear
   inputs (parameters) and outputs (`return`).
2. Cleaning fixes what it can and **rejects with a reason** what it cannot. Never drop rows silently.
3. Test each stage on its own with tiny hand-made data you can check in your head.
4. Stages return; only `main()` prints and saves. `main()` should read like the plan.

**Next time:** the Python capstone begins. You will plan your own application the same way: stages, modules,
functions and tests, *before* writing the code.